In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load your cleaned dataset
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2023": "pop_2023",
    "Established firms 1989": "firms_1989",
    "Established firms 2022": "firms_2022"
})

# Convert relevant columns to numeric (including from "DNE" or other junk)
df['church_density_1890'] = pd.to_numeric(df['church_density_1890'], errors='coerce')
df['income_1989_real_2023'] = pd.to_numeric(df['income_1989_real_2023'], errors='coerce')
df['pct_hs_1990'] = pd.to_numeric(df['pct_hs_1990'], errors='coerce')
df['pop_2023'] = pd.to_numeric(df['pop_2023'], errors='coerce')
df['firms_2022'] = pd.to_numeric(df['firms_2022'], errors='coerce')
df['firms_1989'] = pd.to_numeric(df['firms_1989'], errors='coerce')

# Drop rows that would break the log transforms
df = df[(df['firms_2022'] > 0) & (df['pop_2023'] > 0) & (df['firms_1989'] > 0)]

# Apply log transforms
df['log_firms_2022'] = np.log(df['firms_2022'])
df['log_firms_1989'] = np.log(df['firms_1989'])
df['log_pop_2023'] = np.log(df['pop_2023'])

# Standardize controls
scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real_2023', 'pct_hs_1990']]
)

# Final dataset for regression: filter only once, now that all transforms are done
df_reg = df.dropna(subset=[
    'church_density_1890',
    'income_1989_scaled',
    'pct_hs_1990_scaled',
    'log_firms_1989',
    'log_pop_2023',
    'log_firms_2022',
    'State'
])
df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna()

# Optional: see how many rows are used
print(f"Number of observations used in regression: {len(df_reg)}")

# Run the regression
model = smf.ols(
    formula='log_firms_2022 ~ church_density_1890 + income_1989_scaled + pct_hs_1990_scaled + log_firms_1989 + log_pop_2023 + C(State)',
    data=df_reg
).fit(cov_type='HC3')

# Output the summary
print(model.summary())


Number of observations used in regression: 2610
                            OLS Regression Results                            
Dep. Variable:         log_firms_2022   R-squared:                       0.965
Model:                            OLS   Adj. R-squared:                  0.964
Method:                 Least Squares   F-statistic:                     1412.
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        11:26:04   Log-Likelihood:                -419.67
No. Observations:                2610   AIC:                             943.3
Df Residuals:                    2558   BIC:                             1248.
Df Model:                          51                                         
Covariance Type:                  HC3                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------